# Lab 3.1 — Introduction to Graph Data
**Module III · Graph Neural Networks**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanielFPerez/llm-gnns-course_solutions/blob/main/module-3-gnn/lab3_1_graph_data.ipynb)

---

## What you will do
1. Build and explore small graphs with **NetworkX** — the Swiss army knife of graph analysis in Python.
2. Load the **Cora citation network** via PyTorch Geometric and understand its structure.
3. Visualise **degree distributions** and **node class distributions** — the analogue of EDA from Module I.
4. Understand how PyG represents a graph as a **Data object** with node features, edge indices, labels, and split masks.
5. Explore a **multi-graph dataset** from the biomedical domain (protein graph classification) and see how PyG batches many small graphs together.
6. Build a toy **heterogeneous graph** from a relational database, in the spirit of Stanford/Kumo's Relational Deep Learning and RelBench, using PyG's `HeteroData`.
7. `[Extension]` Compute basic graph statistics (diameter, clustering coefficient, connected components).

## Prerequisites
Module I and II completed. Basic Python comfort.

**Estimated time:** 70–85 min

---
## 0 · Setup

In [ ]:
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running in Google Colab. Cloning the course solutions repository and installing dependencies...")
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/DanielFPerez/llm-gnns-course_solutions.git"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r",
         "llm-gnns-course_solutions/environment/requirements.txt"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "torch-geometric"], check=True)
    sys.path.insert(0, "llm-gnns-course_solutions")
else:
    sys.path.insert(0, str(Path("..").resolve()))

print("Setup complete.")

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import torch

from utils import plot_graph, plot_degree_distribution, check_graph

print("NetworkX", nx.__version__)
print("PyTorch", torch.__version__)
print("Imports OK.")

---
## 1 · Graphs in Python with NetworkX

A **graph** $G = (V, E)$ is a set of **nodes** $V$ connected by **edges** $E$. NetworkX is the standard Python library for creating, manipulating, and analysing graphs.

| NetworkX type | Directed? | Self-loops? | Use when... |
|---|---|---|---|
| `nx.Graph` | No | Yes | Friendships, co-authorship |
| `nx.DiGraph` | Yes | Yes | Citations, web links, follower graphs |
| `nx.MultiGraph` | No | Yes | Multiple edge types between same pair |

```python
G = nx.Graph()
G.add_nodes_from([0, 1, 2, 3])
G.add_edges_from([(0, 1), (1, 2), (2, 3), (3, 0)])
```

### Exercise 3.1.1 `[Core]` — Build a small citation graph

Build an **undirected** graph representing the following mini citation network of 6 papers.

| Paper ID | Title (short) | Cites |
|---|---|---|
| 0 | Deep Learning Survey | 1, 2 |
| 1 | Backpropagation | — |
| 2 | Convolutional Nets | 1, 3 |
| 3 | ImageNet | 1 |
| 4 | Graph Networks | 2, 5 |
| 5 | Spectral GCN | 3 |

1. Create `G` as an `nx.Graph`.
2. Add nodes 0–5 with a `title` attribute.
3. Add the citation edges from the table above.
4. Print the number of nodes and edges.
5. Call `check_graph("3.1.1", G, min_nodes=6, min_edges=6)`.

In [ ]:
# --- SOLUTION ---
titles = [
    "Deep Learning Survey",
    "Backpropagation",
    "Convolutional Nets",
    "ImageNet",
    "Graph Networks",
    "Spectral GCN",
]

G = nx.Graph()
for i, title in enumerate(titles):
    G.add_node(i, title=title)

edges = [(0, 1), (0, 2), (2, 1), (2, 3), (3, 1), (4, 2), (4, 5), (5, 3)]
G.add_edges_from(edges)

print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
check_graph("3.1.1", G, min_nodes=6, min_edges=6)

### Exercise 3.1.2 `[Core]` — Visualise the graph

1. Draw `G` with `plot_graph`, sizing each node by its **degree** — bigger circle = more citations. Build a `node_size` dict (e.g. `{node: 300 + 200 * degree}`) and pass it via `plot_graph`'s `node_size` argument.
2. Answer in a markdown cell below:

- Which node has the highest **degree** (most connections)?
- What does high degree represent in a citation network?

In [ ]:
# --- SOLUTION ---
degrees = dict(G.degree())
node_sizes = {n: 300 + 200 * d for n, d in degrees.items()}

plot_graph(G, node_size=node_sizes, title="Mini citation network (node size ∝ degree)")
plt.show()

max_node = max(degrees, key=degrees.get)
print(f"Node with highest degree: {max_node} ({titles[max_node]}) — degree {degrees[max_node]}")

> **Answer:**
> - Node 1 (Backpropagation) has the highest degree (degree 3) — it is cited by Deep Learning Survey, Convolutional Nets, and ImageNet.
> - In a citation network, high degree (in-degree specifically) indicates an **influential paper** — one that many others build upon. Backpropagation is indeed foundational to most deep learning work.

---
## 2 · Graph properties and degree distributions

The **degree** of a node is its number of direct neighbours. The degree distribution tells us the "shape" of connectivity in the graph:

- **Regular graph**: all nodes have the same degree.
- **Erdős-Rényi random graph**: Poisson-distributed degrees.
- **Scale-free / power-law graph**: a few hubs have very high degree, most nodes have few connections. Many real-world networks (the Web, citation networks, social graphs) are approximately scale-free.

### Exercise 3.1.3 `[Core]` — Degree distribution

1. Extract the degree sequence of `G` as a Python list.
2. Print the minimum, maximum, and mean degree.
3. Plot it with `plot_degree_distribution(degrees, title="Mini citation network — degree distribution")`.

In [ ]:
# --- SOLUTION ---
degrees_list = [d for _, d in G.degree()]

print(f"Min degree : {min(degrees_list)}")
print(f"Max degree : {max(degrees_list)}")
print(f"Mean degree: {np.mean(degrees_list):.2f}")

plot_degree_distribution(degrees_list, title="Mini citation network — degree distribution")
plt.show()

---
## 3 · The Cora citation network

**Cora** is the "MNIST of graph learning" — the standard benchmark used in almost every GNN paper.

| Property | Value |
|---|---|
| Nodes | 2,708 papers |
| Edges | 5,429 citation links |
| Node features | 1,433 binary bag-of-words |
| Classes | 7 research topics |
| Train / Val / Test | 140 / 500 / 1,000 |

Each node is a scientific paper. Each edge is a citation. Each node has a 1,433-dimensional binary feature vector (one entry per vocabulary word: 1 if the word appears in the abstract, 0 otherwise). The task is **node classification**: predict the research topic of each paper.

PyTorch Geometric represents a graph as a `Data` object:

```python
data.x           # (N, F) node feature matrix
data.edge_index  # (2, E) edge list — each column is [src, dst]
data.y           # (N,)   integer class label per node
data.train_mask  # (N,)   boolean — True for training nodes
data.val_mask    # (N,)   boolean — True for validation nodes
data.test_mask   # (N,)   boolean — True for test nodes
```

In [ ]:
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T

dataset = Planetoid(root="/tmp/Cora", name="Cora",
                    transform=T.NormalizeFeatures())
data = dataset[0]

print(dataset)
print(f"Number of graphs  : {len(dataset)}")
print(f"Number of features: {dataset.num_features}")
print(f"Number of classes : {dataset.num_classes}")

### Exercise 3.1.4 `[Core]` — Explore the Data object

Print the following facts about `data`:
1. Shape of `data.x` (node features).
2. Shape of `data.edge_index`.
3. Number of training, validation, and test nodes.
4. The integer class of the first training node.

In [ ]:
# --- SOLUTION ---
print(f"Node features  x : {data.x.shape}  (nodes × features)")
print(f"Edge index       : {data.edge_index.shape}  (2 × edges)")
print(f"Train nodes      : {data.train_mask.sum().item()}")
print(f"Val   nodes      : {data.val_mask.sum().item()}")
print(f"Test  nodes      : {data.test_mask.sum().item()}")

first_train_node = data.train_mask.nonzero(as_tuple=True)[0][0].item()
print(f"Class of first train node (node {first_train_node}): {data.y[first_train_node].item()}")

### Exercise 3.1.5 `[Core]` — Cora class distribution

Plot a bar chart showing how many nodes belong to each of the 7 research topic classes.

Use `data.y.numpy()` to get the labels as a NumPy array. You can use `np.unique(labels, return_counts=True)` or pandas `value_counts`. Label the x-axis with the class names below.

```python
CLASS_NAMES = [
    "Case-Based", "Genetic Algorithms", "Neural Networks",
    "Probabilistic Methods", "Reinforcement Learning",
    "Rule Learning", "Theory",
]
```

Is the dataset balanced?

In [ ]:
# --- SOLUTION ---
CLASS_NAMES = [
    "Case-Based", "Genetic Algorithms", "Neural Networks",
    "Probabilistic Methods", "Reinforcement Learning",
    "Rule Learning", "Theory",
]

labels = data.y.numpy()
classes, counts = np.unique(labels, return_counts=True)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(CLASS_NAMES, counts, color="#4C72B0", edgecolor="white")
ax.set_xlabel("Research topic")
ax.set_ylabel("Number of papers")
ax.set_title("Cora — node class distribution")
ax.tick_params(axis="x", rotation=30)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print("Counts per class:")
for name, count in zip(CLASS_NAMES, counts):
    print(f"  {name:<25}: {count:>4}  ({count/len(labels):.1%})")

> **Key observation:** Cora is **moderately imbalanced** — Neural Networks is by far the largest class (~600 papers, ~22%), while Rule Learning is the smallest (~180 papers, ~7%). This is typical of real citation networks: most papers belong to a dominant subfield.

### Exercise 3.1.6 `[Core]` — Cora degree distribution

Compute and plot the degree distribution of the Cora graph.

**Hint:** PyG's `edge_index` is a `(2, E)` tensor where `edge_index[0]` holds source nodes. You can get the degree of each node by counting how often each node ID appears:

```python
from torch_geometric.utils import degree
deg = degree(data.edge_index[0], num_nodes=data.num_nodes)
```

After plotting:
- What is the maximum degree?
- Does the distribution look like a power law (long tail with a few very high-degree hubs)?

In [ ]:
# --- SOLUTION ---
from torch_geometric.utils import degree

deg = degree(data.edge_index[0], num_nodes=data.num_nodes).numpy().astype(int)

print(f"Min degree : {deg.min()}")
print(f"Max degree : {deg.max()}")
print(f"Mean degree: {deg.mean():.2f}")
print(f"Median degree: {np.median(deg):.0f}")

plot_degree_distribution(deg.tolist(), title="Cora — degree distribution")
plt.show()

# Log-log plot to check for power law
uniq, cnts = np.unique(deg, return_counts=True)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(uniq, cnts, s=15, alpha=0.7, color="#4C72B0")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Degree (log scale)")
ax.set_ylabel("Count (log scale)")
ax.set_title("Cora — degree distribution (log-log)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

> **Key observation:** The log-log plot shows an approximately linear relationship — the hallmark of a **power-law / scale-free network**. Most papers cite just 1–3 others, but a small number of foundational papers are cited by dozens. This heavy-tailed structure is why GNNs must handle high-degree hubs gracefully (which is part of why GAT's per-edge attention is appealing: it can down-weight uninformative neighbours even for high-degree nodes).

---
## 4 · Converting between NetworkX and PyG

Sometimes it is useful to convert between PyG's `Data` representation and NetworkX for visualisation or analysis. PyG provides utilities for this:

```python
from torch_geometric.utils import to_networkx
G_nx = to_networkx(data, to_undirected=True)
```

### Exercise 3.1.7 `[Core]` — Convert Cora to NetworkX and sample a subgraph

1. Convert `data` to a NetworkX graph.
2. Print the number of nodes and edges (should match `data.num_nodes` and `data.num_edges // 2`).
3. Extract the **ego graph** of node 0 (node 0 and all its 1-hop neighbours) using `nx.ego_graph(G, 0, radius=1)`.
4. Draw the ego graph with `plot_graph`, colouring nodes by their Cora class label.

**Hint:** Build a `labels` dict mapping node id → integer class from `data.y`.

In [ ]:
# --- SOLUTION ---
from torch_geometric.utils import to_networkx

G_cora = to_networkx(data, to_undirected=True)
print(f"NetworkX graph: {G_cora.number_of_nodes()} nodes, {G_cora.number_of_edges()} edges")

ego = nx.ego_graph(G_cora, 0, radius=1)
print(f"Ego graph of node 0: {ego.number_of_nodes()} nodes, {ego.number_of_edges()} edges")

node_labels = {i: data.y[i].item() for i in ego.nodes()}
plot_graph(
    ego,
    labels=node_labels,
    label_names=CLASS_NAMES,
    title=f"Ego graph of node 0 (class: {CLASS_NAMES[data.y[0].item()]})",
    node_size=300,
)
plt.show()

> **Notice:** Node 0's immediate neighbours are not all the same class — that is exactly the challenge. A GNN must learn to aggregate neighbourhood information to classify node 0 correctly. In Module 3.2 we will train a GCN that does exactly this — and we will see that the graph structure carries far more discriminative signal than the raw features alone.

---
## 5 · Beyond a single graph: multi-graph datasets

Cora is **one** large graph, and the task is **node classification** — predict a label for each node inside that single graph. Many real problems instead hand you a **dataset of many separate, small graphs**, each with its own label, and the task becomes **graph classification** — predict a label for the whole graph.

The classic domain for this is **bioinformatics**: a molecule or a protein is naturally a graph (atoms/residues = nodes, bonds/contacts = edges), and the label is a property of the whole structure (toxic vs. not, enzyme vs. not, active vs. inactive against a target).

We'll use **PROTEINS**, a standard graph-classification benchmark bundled with PyTorch Geometric through `TUDataset`. Each of its 1,113 graphs represents one protein: nodes are secondary structure elements (helices, sheets, turns), edges connect elements that are close in the amino-acid sequence or in 3-D space, and the binary label says whether the protein is an **enzyme**.

### Exercise 3.1.8 `[Core]` — Load and explore a multi-graph dataset

1. Load `TUDataset(root="/tmp/TUDataset", name="PROTEINS")`.
2. Print the number of graphs, `dataset.num_features`, and `dataset.num_classes`.
3. Print the number of nodes, edges, and label of the **first** graph in the dataset.
4. Compute the average number of nodes and edges **per graph** across the whole dataset.
5. Print the class balance (how many enzymes vs. non-enzymes).

In [ ]:
# --- SOLUTION ---
from torch_geometric.datasets import TUDataset

proteins = TUDataset(root="/tmp/TUDataset", name="PROTEINS")

print(proteins)
print(f"Number of graphs  : {len(proteins)}")
print(f"Number of features: {proteins.num_features}")
print(f"Number of classes : {proteins.num_classes}")

g0 = proteins[0]
print(f"\nFirst graph: {g0.num_nodes} nodes, {g0.num_edges} edges, label={g0.y.item()}")

num_nodes_per_graph = [g.num_nodes for g in proteins]
num_edges_per_graph = [g.num_edges for g in proteins]
print(f"\nAvg nodes/graph: {np.mean(num_nodes_per_graph):.1f} (min {min(num_nodes_per_graph)}, max {max(num_nodes_per_graph)})")
print(f"Avg edges/graph: {np.mean(num_edges_per_graph):.1f}")

labels = np.array([g.y.item() for g in proteins])
classes, counts = np.unique(labels, return_counts=True)
print(f"\nClass balance (0=non-enzyme, 1=enzyme): {dict(zip(classes.tolist(), counts.tolist()))}")

> **Key observation:** Unlike Cora, where every node lives inside the *same* fixed graph, PROTEINS gives us 1,113 *independent* graphs of wildly different sizes (from a handful of nodes to several hundred). There is no shared node-id space across graphs — graph 5's node 0 has nothing to do with graph 12's node 0. This is why graph classification needs a different data-loading strategy than node classification, which we explore next.

### Exercise 3.1.9 `[Core]` — Visualise a few protein graphs and understand batching

1. Find the indices of the 3 **smallest** graphs (fewest nodes) in `proteins`.
2. Draw each of them with `plot_graph`, using `to_networkx`, colouring each graph with `node_color` (e.g. one colour for enzymes, another for non-enzymes) and titling each plot with its label and node count.
3. Build a `torch_geometric.loader.DataLoader` over `proteins` with `batch_size=32` and pull one mini-batch. Print it, along with `batch.batch.shape` and `batch.num_graphs`.

**Hint:** `DataLoader` merges the graphs in a mini-batch into a single large graph with disconnected components, and creates a `batch` vector that maps every node back to the graph it came from — that's what lets a GNN process many graphs at once and later pool node embeddings back into one embedding per graph.

In [ ]:
# --- SOLUTION ---
from torch_geometric.utils import to_networkx
from torch_geometric.loader import DataLoader

order = np.argsort(num_nodes_per_graph)
smallest_idx = order[:3]

ENZYME_COLORS = {0: "#4C72B0", 1: "#DD8452"}  # 0=non-enzyme, 1=enzyme

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, idx in zip(axes, smallest_idx):
    g = proteins[int(idx)]
    g_nx = to_networkx(g, to_undirected=True)
    label_str = "enzyme" if g.y.item() == 1 else "non-enzyme"
    plot_graph(
        g_nx,
        node_color=ENZYME_COLORS[g.y.item()],
        title=f"Graph {idx} — {label_str} ({g.num_nodes} nodes)",
        ax=ax,
    )
plt.tight_layout()
plt.show()

loader = DataLoader(proteins, batch_size=32, shuffle=True)
batch = next(iter(loader))

print(batch)
print(f"\nbatch.batch shape: {tuple(batch.batch.shape)} — maps each node to its graph's index within the mini-batch")
print(f"Number of graphs in this mini-batch: {batch.num_graphs}")

> **Key observation:** `batch.batch` is a vector of length `batch.num_nodes` whose values run from `0` to `batch.num_graphs - 1`. A graph-level GNN runs its usual message-passing layers on this single disconnected super-graph, then uses `batch` with a **readout / global pooling** operation (e.g. `global_mean_pool`) to collapse each graph's node embeddings into one vector — that vector is what gets classified. This is the same PyG `Data`/`DataLoader` machinery you already know from Cora, extended with one extra bookkeeping tensor. We won't train a graph classifier in this course, but the data pipeline generalises directly from what you've just seen.

---
## 6 · Heterogeneous graphs: from relational databases to graphs

Every graph so far — the citation network, Cora, PROTEINS — is **homogeneous**: one node type, one edge type. Real-world data usually isn't. It lives in **relational databases**: a `users` table, a `products` table, an `orders` table, linked by foreign keys.

Converting a relational database into a graph is exactly the idea behind **Relational Deep Learning (RDL)**, an approach developed by a Stanford group (Jure Leskovec's lab) together with **Kumo.ai**, and packaged into the open **RelBench** benchmark (relbench.stanford.edu). The recipe:

- Each **table** becomes a **node type** — its rows are nodes, its non-key columns become that node type's features.
- Each **foreign key** becomes an **edge type** connecting the two node types it links.
- The result is a **heterogeneous graph**: more than one node type and/or edge type, which PyG represents with `HeteroData` instead of a plain `Data` object.

Below we build a miniature version of this idea by hand — a toy e-commerce database with `User`, `Product`, and `Order` tables — so you can see exactly how "tables → graph" works before ever touching a real benchmark.

> **Going further:** RelBench ships real relational databases (e-commerce, sports statistics, medical records, forums, ...) already converted into `HeteroData` graphs, with predictive tasks like *"will this customer churn?"* or *"will this user buy again in the next 7 days?"*. It's a natural next step once you're comfortable with the mechanics practiced here (`pip install relbench` — not part of this course's `requirements.txt`).

### Exercise 3.1.10 `[Core]` — Build a toy relational database

Represent a tiny e-commerce database as three pandas DataFrames:

- `users_df`: `user_id`, `age`, `loyalty_tier` (0 = bronze, 1 = silver, 2 = gold)
- `products_df`: `product_id`, `category` (0 = electronics, 1 = books, 2 = home), `price`
- `orders_df`: `order_id`, `user_id` (FK → users), `product_id` (FK → products), `quantity`

Use 5 users, 6 products, and 8 orders (values of your choice, or the ones in the solution). Print each table's shape and its first few rows.

In [ ]:
# --- SOLUTION ---
import pandas as pd

users_df = pd.DataFrame({
    "user_id": [0, 1, 2, 3, 4],
    "age": [25, 34, 45, 22, 31],
    "loyalty_tier": [0, 1, 1, 0, 2],  # 0=bronze, 1=silver, 2=gold
})

products_df = pd.DataFrame({
    "product_id": [0, 1, 2, 3, 4, 5],
    "category": [0, 0, 1, 1, 2, 2],  # 0=electronics, 1=books, 2=home
    "price": [299.0, 49.0, 15.0, 22.5, 89.0, 120.0],
})

orders_df = pd.DataFrame({
    "order_id":   [0, 1, 2, 3, 4, 5, 6, 7],
    "user_id":    [0, 0, 1, 2, 2, 3, 4, 4],
    "product_id": [0, 2, 1, 3, 4, 5, 0, 2],
    "quantity":   [1, 2, 1, 1, 3, 1, 1, 2],
})

for name, df in [("users", users_df), ("products", products_df), ("orders", orders_df)]:
    print(f"{name}: {df.shape}")
    print(df, "\n")

### Exercise 3.1.11 `[Core]` — Convert the tables into a PyG `HeteroData` graph

1. Create an empty `torch_geometric.data.HeteroData()`, call it `data_hetero` (to avoid confusion with the Cora `data` object from Section 3).
2. Set node features per type: `data_hetero["user"].x`, `data_hetero["product"].x`, `data_hetero["order"].x` (use the non-key columns of each table as float tensors).
3. Add the two foreign-key relationships as edge types: `("user", "places", "order")` and `("order", "contains", "product")`.
4. Apply `torch_geometric.transforms.ToUndirected()` so information can also flow "backwards" along each relation.
5. Print `data_hetero`, `data_hetero.node_types`, and `data_hetero.edge_types`.

In [ ]:
# --- SOLUTION ---
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T

data_hetero = HeteroData()

data_hetero["user"].x = torch.tensor(users_df[["age", "loyalty_tier"]].values, dtype=torch.float)
data_hetero["product"].x = torch.tensor(products_df[["category", "price"]].values, dtype=torch.float)
data_hetero["order"].x = torch.tensor(orders_df[["quantity"]].values, dtype=torch.float)

data_hetero["user", "places", "order"].edge_index = torch.tensor(
    np.array([orders_df["user_id"].values, orders_df["order_id"].values]), dtype=torch.long
)
data_hetero["order", "contains", "product"].edge_index = torch.tensor(
    np.array([orders_df["order_id"].values, orders_df["product_id"].values]), dtype=torch.long
)

data_hetero = T.ToUndirected()(data_hetero)

print(data_hetero)
print(f"\nNode types: {data_hetero.node_types}")
print(f"Edge types: {data_hetero.edge_types}")

> **Key observation:** `data_hetero` now stores three feature matrices of *different shapes and meanings* (`user.x` is `[5, 2]`, `product.x` is `[6, 2]`, `order.x` is `[8, 1]`) plus four edge types (two original + two reverse, added by `ToUndirected`). A plain GCN layer can't be applied here — it assumes one shared feature space and one adjacency matrix. Heterogeneous GNNs (PyG's `HeteroConv`, `to_hetero()`, or `HGTConv`) instead learn a **separate transformation per edge type**, then combine the messages arriving at each node type. This is exactly the model family RelBench and Kumo use to make predictions directly on relational data, without hand-engineering features across tables.

### Exercise 3.1.12 `[Core]` — Visualise the heterogeneous graph

1. Draw the whole toy database as one coloured graph: each node type gets its own colour, using `plot_graph`'s `labels`/`label_names` arguments.
2. Instead of the default spring layout, build an explicit `pos` dict that places `user` nodes in a left column, `order` nodes in a middle column, and `product` nodes in a right column, and pass it via `plot_graph`'s `pos` argument — so the relational structure (user → order → product) is visually obvious.

**Hint:** `plot_graph` expects a single `networkx.Graph` with one shared, global node-id space, but `HeteroData` indexes each node type from 0 independently (`user` 0–4, `product` 0–5, `order` 0–7). Assign each node type a contiguous block of global ids, build the combined `nx.Graph`, and pass `labels={global_id: type_index}`, `label_names=data_hetero.node_types`, `pos={global_id: (x, y)}`.

In [ ]:
# --- SOLUTION ---
offset, type_offset = 0, {}
for ntype in data_hetero.node_types:
    type_offset[ntype] = offset
    offset += data_hetero[ntype].num_nodes

type_to_idx = {ntype: i for i, ntype in enumerate(data_hetero.node_types)}
column_x = {"user": 0.0, "order": 1.0, "product": 2.0}

G_hetero = nx.Graph()
node_type_label, pos = {}, {}
for ntype in data_hetero.node_types:
    for local_id in range(data_hetero[ntype].num_nodes):
        gid = type_offset[ntype] + local_id
        G_hetero.add_node(gid)
        node_type_label[gid] = type_to_idx[ntype]
        pos[gid] = (column_x[ntype], -local_id)

for (src_type, _relation, dst_type), store in data_hetero.edge_items():
    for s, d in zip(*store.edge_index.tolist()):
        G_hetero.add_edge(type_offset[src_type] + s, type_offset[dst_type] + d)

plot_graph(
    G_hetero,
    labels=node_type_label,
    label_names=list(data_hetero.node_types),
    pos=pos,
    title="Toy e-commerce database as a heterogeneous graph",
    node_size=250,
)
plt.show()

> **Notice:** With this columnar layout, `order` nodes visibly sit "between" `user` and `product` nodes — mirroring the foreign-key structure of the original tables. Scale this picture up to real databases with dozens of tables and millions of rows, keep the edges **temporal** (only connect events available at prediction time), and you have the setting RelBench and Kumo target: turning an entire production database into a graph a GNN can learn from directly.

---
## 7 · `[Extension]` Graph statistics

### Exercise 3.1.13 `[Extension]` — Basic graph statistics on Cora

Using NetworkX, compute the following properties of the full Cora graph:

1. **Number of connected components** — is the graph fully connected?
2. **Average clustering coefficient** — do neighbours of a node tend to be connected to each other?
3. **Diameter** of the largest connected component — the longest shortest path.

**Note:** For large graphs, computing the exact diameter is slow. Use `nx.approximation.diameter` or sample 100 random nodes and take the maximum shortest-path length among them.

Interpret the results.

In [ ]:
# --- SOLUTION ---
n_components = nx.number_connected_components(G_cora)
print(f"Connected components: {n_components}")

avg_clustering = nx.average_clustering(G_cora)
print(f"Average clustering coefficient: {avg_clustering:.4f}")

# Approximate diameter on the largest component
largest_cc = G_cora.subgraph(
    max(nx.connected_components(G_cora), key=len)
).copy()
print(f"Largest component: {largest_cc.number_of_nodes()} nodes")

# Sample-based approximate diameter
rng = np.random.default_rng(42)
sample_nodes = rng.choice(list(largest_cc.nodes()), size=50, replace=False)
max_dist = 0
for src in sample_nodes:
    lengths = nx.single_source_shortest_path_length(largest_cc, src)
    max_dist = max(max_dist, max(lengths.values()))
print(f"Approximate diameter (sampled): {max_dist}")

> **Key observations:**
> - Cora has a handful of isolated components, but the vast majority of nodes are in one **giant connected component** — typical of real citation networks.
> - A **clustering coefficient ≈ 0.24** means neighbours of a node are 24% likely to also be connected to each other — much higher than a random graph of the same density, indicating **community structure** (papers in the same subfield cite each other).
> - A **diameter of ~19** means that every paper can reach any other in the giant component in at most ~19 citation hops — the "small world" property. This also implies that a 2-layer GCN (which only aggregates 2-hop neighbours) can only directly see a tiny fraction of the graph from any given node — which motivates deeper architectures or global attention mechanisms (Graph Transformers, Module IV).

---
## Summary

| What we learned | Key takeaway |
|---|---|
| NetworkX creates and manipulates graphs in Python | Standard tool for small-to-medium graphs and analysis |
| PyG `Data` object stores features, edges, labels, masks | The node feature matrix + edge index is everything a GNN needs |
| Cora is the benchmark citation network | 2,708 nodes, 1,433 features, 7 classes, semi-supervised setting |
| Degree distributions are heavy-tailed | A few hub papers are cited by many — GNNs must handle this |
| Community structure is strong | Local neighbourhoods are informative → message passing works |
| Multi-graph datasets (e.g. PROTEINS) need graph classification | PyG `DataLoader` batches many graphs into one disjoint graph + a `batch` vector |
| Relational databases become heterogeneous graphs | Tables → node types, foreign keys → edge types (`HeteroData`, RelBench/RDL) |

**Next → Lab 3.2:** We train an MLP (ignoring the graph) vs a GCN (using the graph) on Cora and measure how much the graph structure is worth.